# Pipeline de Detecção da Bola com Pré‑Processamento

Este notebook implementa um pipeline completo para a detecção da bola em vídeos de partidas. Ele abrange todas as etapas necessárias, desde a preparação dos dados e pré‑processamento de imagens até o treinamento e otimização do modelo YOLO utilizando um Algoritmo Genético.

### Resumo das Etapas
- **Etapas Fora do Colab**: Corte do vídeo da partida, upload no Roboflow e criação/compactação do dataset.
- Extração e limpeza dos frames dos vídeos.
- Aplicação de técnicas de pré‑processamento (equalização de histograma, CLAHE, correção gamma, remoção de fundo).
- Criação do arquivo YAML para a configuração do dataset (treino/validação).
- Treinamento do modelo YOLO e otimização dos parâmetros via Algoritmo Genético (DEAP).

Adapte os caminhos, parâmetros e funções conforme as necessidades do seu projeto.

## 1. Etapas Fora do Ambiente do Colab

Antes de executar o pipeline completo no Colab, realize as seguintes etapas **fora** do ambiente:

1. **Corte do Vídeo da Partida usando Shotut**:
   - Utilize o Shotut para cortar o trecho da partida que será utilizado para a análise.

2. **Upload do Trecho no Roboflow**:
   - Faça o upload do trecho cortado no Roboflow para que seja realizada a extração dos frames, definição dos roundboxes e criação do dataset.

3. **Upload do Dataset**:
   - Após a criação do dataset, compacte-o em um arquivo `dataset.zip` e faça o upload conforme necessário.

In [ ]:
# Setup Inicial: Instalação, importação e definição de diretórios

!pip install --upgrade ultralytics
!pip install deap

import os
import shutil
import glob
import random
import time
import cv2
import numpy as np
import pandas as pd
import logging
import json
import torch
import torch.nn.functional as F
import math
from ultralytics import YOLO
from deap import base, creator, tools, algorithms
from joblib import Parallel, delayed

# Diretórios base (no Colab, normalmente '/content')
BASE_DIR = '/content'
VIDEOS_RAW_DIR = os.path.join(BASE_DIR, 'videos_raw')
FRAMES_EXTRAIDOS_DIR = os.path.join(BASE_DIR, 'videos_extraidos')
FRAMES_LIMPOS_DIR = os.path.join(BASE_DIR, 'frames_limpos')
LABELS_DIR = os.path.join(FRAMES_LIMPOS_DIR, 'labels')

# Diretórios para datasets (original e pré‑processados)
DATASET_ORIGINAL = os.path.join(BASE_DIR, 'dataset_original')
DATASET_HIST = os.path.join(BASE_DIR, 'dataset_hist')
DATASET_CLAHE = os.path.join(BASE_DIR, 'dataset_clahe')
DATASET_GAMMA = os.path.join(BASE_DIR, 'dataset_gamma')
DATASET_REMOVEBG = os.path.join(BASE_DIR, 'dataset_removebg')

# Configuração do log para registrar as avaliações do GA
logging.basicConfig(
    filename="ga_optimization.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Parâmetros globais
FRAME_RATE = 1  # 1 frame por segundo

# Criação dos diretórios básicos
for d in [VIDEOS_RAW_DIR, FRAMES_EXTRAIDOS_DIR, FRAMES_LIMPOS_DIR, LABELS_DIR,
          DATASET_ORIGINAL, DATASET_HIST, DATASET_CLAHE, DATASET_GAMMA, DATASET_REMOVEBG]:
    os.makedirs(d, exist_ok=True)

# Baixar o modelo YOLO (caso não exista) e instalar a biblioteca ultralytics
model_path_small = os.path.join(BASE_DIR, "yolov11-small.pt")
if not os.path.exists(model_path_small):
    !wget -O yolov11-small.pt "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11s.pt"

model_path_large = os.path.join(BASE_DIR, "yolov11-large.pt")
if not os.path.exists(model_path_large):
    !wget -O yolov11-large.pt "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11l.pt"

In [ ]:
!unzip -q dataset.zip

## Parâmetros Principais

Aqui definimos os principais parâmetros que serão utilizados tanto no **Algoritmo Genético (GA)** quanto no **treinamento final** do modelo YOLO:

- **GA_POPULATION**: Tamanho da população do GA.
- **GA_GENERATIONS**: Número de gerações para evolução do GA.
- **GA_EPOCHS**: Número de épocas (epochs) utilizadas no treinamento YOLO dentro do GA (processo de avaliação).
- **TRAINING_EPOCHS**: Número de épocas para o treinamento final do modelo YOLO, após a escolha dos melhores parâmetros.


In [ ]:
# Definição dos parâmetros principais

GA_POPULATION = 10      # Tamanho da população do GA
GA_GENERATIONS = 10     # Número de gerações do GA
GA_EPOCHS = 10          # Épocas de treinamento YOLO durante a avaliação do GA
TRAINING_EPOCHS = 50   # Épocas de treinamento YOLO após encontrar os melhores parâmetros

print("Parâmetros definidos:")
print(f"GA_POPULATION = {GA_POPULATION}")
print(f"GA_GENERATIONS = {GA_GENERATIONS}")
print(f"GA_EPOCHS = {GA_EPOCHS}")
print(f"TRAINING_EPOCHS = {TRAINING_EPOCHS}")

## 2. Funções de Pré‑Processamento

Nesta seção são definidas funções para:
- **Extração da bounding box da bola**: `extract_ball_box` e `get_roi_from_bbox`.
- **Aplicação de técnicas de pré‑processamento**: equalização de histograma, CLAHE, correção gamma e remoção de fundo com a função `apply_pipeline`.

Também são disponibilizadas funções específicas para remoção de fundo, equalização de histograma, CLAHE e correção gamma.

In [ ]:
def extract_ball_box(results, img):
    """Extrai a bounding box da bola a partir dos resultados da predição, considerando um limiar mínimo de confiança."""
    threshold = 0.3
    result = results[0] if isinstance(results, list) else results
    for box in result.boxes:
        conf = box.conf.item() if hasattr(box, "conf") else 1.0
        if conf < threshold:
            continue
        xywh = box.xywh[0]
        xywh = xywh.cpu().numpy() if hasattr(xywh, "cpu") else np.array(xywh)
        return float(xywh[0]), float(xywh[1]), float(xywh[2]), float(xywh[3])
    return None

def get_roi_from_bbox(bbox, image_shape, roi_scale=3.0):
    """Calcula a região de interesse (ROI) com base na bounding box da bola e um fator de escala."""
    height, width = image_shape[:2]
    cx, cy, w, h = bbox
    roi_w = w * roi_scale
    roi_h = h * roi_scale
    left = int(max(0, cx - roi_w / 2))
    top = int(max(0, cy - roi_h / 2))
    right = int(min(width, cx + roi_w / 2))
    bottom = int(min(height, cy + roi_h / 2))
    return {"left": left, "top": top, "right": right, "bottom": bottom}

def apply_pipeline(image, use_hist_eq, use_clahe, clahe_clip, tile_size, use_gamma, gamma_value,
                   use_remove_bg, lower_bg, upper_bg):
    """Aplica o pipeline de pré‑processamento na imagem de acordo com os parâmetros informados."""
    processed_img = image.copy()

    if use_hist_eq:
        for i in range(3):
            processed_img[..., i] = cv2.equalizeHist(processed_img[..., i])

    if use_clahe:
        clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=tile_size)
        for i in range(3):
            processed_img[..., i] = clahe.apply(processed_img[..., i])

    if use_gamma:
        invGamma = 1.0 / gamma_value
        table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        processed_img = cv2.LUT(processed_img, table)

    if use_remove_bg:
        lower = np.array(lower_bg, dtype="uint8")
        upper = np.array(upper_bg, dtype="uint8")
        mask = cv2.inRange(processed_img, lower, upper)
        processed_img[mask > 0] = 0
    return processed_img

In [ ]:
def remove_bg(image, lower_bound=(100, 100, 100), upper_bound=(140, 255, 255)):
    """Remove o fundo (exemplo para fundo azul). Ajuste as faixas HSV conforme necessário."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array(lower_bound), np.array(upper_bound))
    image[mask != 0] = [0, 0, 0]
    return image

def hist_eq(image):
    """Aplica equalização de histograma na imagem."""
    img_yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    img_yuv[:,:,0] = cv2.equalizeHist(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def clahe(image, clip_limit=2.0, tile_grid_size=(8,8)):
    """Aplica CLAHE (Contrast Limited Adaptive Histogram Equalization) na imagem."""
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe_obj = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    cl = clahe_obj.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

def gamma_correction(image, gamma=1.0):
    """Aplica correção gamma na imagem."""
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype('uint8')
    return cv2.LUT(image, table)


## 3. Criação do Arquivo YAML para o Dataset

Esta etapa cria um arquivo YAML que configura o dataset, definindo os caminhos para as pastas de treino e validação, o número de classes e os nomes das classes (neste caso, apenas a classe `'bola'`).

In [ ]:
def create_yaml_file(dataset_dir, output_yaml_path):
    yaml_content = f"""train: {os.path.join(dataset_dir, 'train')}\nval: {os.path.join(dataset_dir, 'valid')}\nnc: 1\nnames: ['bola']\n"""
    with open(output_yaml_path, 'w') as f:
        f.write(yaml_content)
    print(f"Arquivo YAML criado: {output_yaml_path}")

# Dicionário com as variantes de dataset (aqui, apenas a versão original)
dataset_variants = {
    "Original": DATASET_ORIGINAL
}

yaml_files = {}
for name, dpath in dataset_variants.items():
    yaml_path = os.path.join(dpath, f"{name.lower()}.yaml")
    create_yaml_file(dpath, yaml_path)
    yaml_files[name] = yaml_path

print('Arquivos YAML criados para cada variante.')

## 4. Otimização com Algoritmo Genético (DEAP)

Nesta seção, utilizamos o framework DEAP para otimizar os parâmetros do pipeline de pré‑processamento e treinamento do modelo YOLO.

Os parâmetros otimizados incluem:
- Tipo de pré‑processamento (nenhum, equalização de histograma, CLAHE ou correção gamma).
- Flag para remoção do fundo e os limites para a remoção do fundo (exemplo para fundo azul).
- Parâmetros do CLAHE (clip limit e tamanho do tile).
- Valor de gamma para correção.
- Escala da região de interesse (ROI).

O desempenho de cada configuração é avaliado com base em uma combinação da métrica mAP50 (do treinamento) e da taxa de inferência (fps).

In [ ]:
preproc_options = ['none', 'hist_eq', 'clahe', 'gamma']

ga_evaluations = []

def evaluate_individual(individual):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    preproc_idx, remove_bg_flag, lower_blue, upper_blue, clahe_clip, tile_size, gamma_value, roi_scale = individual
    preproc = preproc_options[int(preproc_idx)]
    remove_bg_flag = bool(remove_bg_flag)

    try:
        model_eval = YOLO("yolov11-small.pt").to(device)
    except Exception as e:
        logging.error("Erro ao carregar o modelo YOLO.", exc_info=True)
        return 0,

    t0 = time.time()
    try:
        results = model_eval.train(
            data="/content/dataset_original/original.yaml",
            epochs=GA_EPOCHS,
            imgsz=1280,
            batch=16,
            device=device,
            augment=True
        )
    except Exception as e:
        logging.error("Erro no treinamento do modelo:", exc_info=True)
        return 0,

    training_time = time.time() - t0
    mAP50 = results.box.map if hasattr(results, "box") and hasattr(results.box, "map") else 0

    val_images = glob.glob("/content/dataset_original/valid/images/*.jpg")
    inference_times = []
    last_ball_bb = None
    batch_tensors = []

    for img_file in val_images:
        img = cv2.imread(img_file)
        if img is None:
            continue

        # Se tivermos uma última posição da bola, focamos primeiro na ROI
        if last_ball_bb is not None:
            roi = get_roi_from_bbox(last_ball_bb, img.shape, roi_scale)
            img = img[roi["top"]:roi["bottom"], roi["left"]:roi["right"]]

        # Aplicar pipeline de pré-processamento
        img = apply_pipeline(
            image=img,
            use_hist_eq=(preproc == 'hist_eq'),
            use_clahe=(preproc == 'clahe'),
            clahe_clip=clahe_clip,
            tile_size=tile_size,
            use_gamma=(preproc == 'gamma'),
            gamma_value=gamma_value,
            use_remove_bg=remove_bg_flag,
            lower_bg=(lower_blue, 100, 100),
            upper_bg=(upper_blue, 255, 255)
        )

        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1).to(device) / 255.0
        batch_tensors.append(img_tensor)

    if not batch_tensors:
        logging.warning("Nenhuma imagem de validação foi carregada corretamente.")
        return 0,

    batch_tensors = torch.stack(batch_tensors)
    torch.cuda.synchronize()
    t_start = time.time()

    results_full = model_eval(batch_tensors)
    torch.cuda.synchronize()
    t_full = time.time() - t_start
    inference_times.append(t_full / len(val_images))

    for i, img_tensor in enumerate(batch_tensors):
        ball_box = extract_ball_box(results_full[i], img_tensor)

        if ball_box is not None:
            last_ball_bb = ball_box  # Atualiza a última posição da bola

    # Se a bola não foi detectada na ROI, repetir a detecção na imagem completa
    if last_ball_bb is None:
        for img_file in val_images:
            img = cv2.imread(img_file)
            if img is None:
                continue

            img = apply_pipeline(
                image=img,
                use_hist_eq=(preproc == 'hist_eq'),
                use_clahe=(preproc == 'clahe'),
                clahe_clip=clahe_clip,
                tile_size=tile_size,
                use_gamma=(preproc == 'gamma'),
                gamma_value=gamma_value,
                use_remove_bg=remove_bg_flag,
                lower_bg=(lower_blue, 100, 100),
                upper_bg=(upper_blue, 255, 255)
            )

            img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1).to(device) / 255.0
            results_full = model_eval(img_tensor.unsqueeze(0))  # Processar imagem única
            ball_box = extract_ball_box(results_full[0], img_tensor)

            if ball_box is not None:
                last_ball_bb = ball_box
                break  # Se encontrar, não precisa continuar procurando

    avg_inference = sum(inference_times) / len(inference_times) if inference_times else 1.0
    fps = 1.0 / avg_inference if avg_inference > 0 else 0
    score = 0.9 * mAP50 + 0.1 * (fps / 30.0)

    individual.details = {
        "preproc": preproc,
        "remove_bg": remove_bg_flag,
        "lower_blue": lower_blue,
        "upper_blue": upper_blue,
        "clahe_clip": clahe_clip,
        "tile_size": tile_size,
        "gamma_value": gamma_value,
        "roi_scale": roi_scale,
        "training_time": training_time,
        "mAP50": mAP50,
        "fps": fps,
        "score": score
    }

    ga_evaluations.append(individual.details)
    logging.info(f"Avaliação - Config: preproc={preproc}, remove_bg={remove_bg_flag}, gamma={gamma_value:.2f} | Score: {score:.4f}")
    print(f"Avaliação - Config: preproc={preproc}, remove_bg={remove_bg_flag}, gamma={gamma_value:.2f} | Score: {score:.4f}")
    return score,

In [ ]:
# Criação dos tipos e registro dos atributos para o Algoritmo Genético

creator.create('FitnessMax', base.Fitness, weights=(1.0,))
creator.create('Individual', list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register('attr_preproc', lambda: random.randint(0, len(preproc_options) - 1))
toolbox.register('attr_remove_bg', lambda: random.randint(0, 1))
toolbox.register('attr_lower_blue', lambda: random.rand int(90, 110))
toolbox.register('attr_upper_blue', lambda: random.randint(130, 160))
toolbox.register('attr_clahe_clip', lambda: random.uniform(1.0, 4.0))
toolbox.register('attr_tile_size', lambda: (random.choice([4, 8, 16]), random.choice([4, 8, 16])))
toolbox.register('attr_gamma_value', lambda: random.uniform(0.8, 1.5))
toolbox.register('attr_roi_scale', lambda: random.uniform(2.0, 5.0))

toolbox.register(
    'individual',
    tools.initCycle,
    creator.Individual,
    (
        toolbox.attr_preproc,
        toolbox.attr_remove_bg,
        toolbox.attr_lower_blue,
        toolbox.attr_upper_blue,
        toolbox.attr_clahe_clip,
        toolbox.attr_tile_size,
        toolbox.attr_gamma_value,
        toolbox.attr_roi_scale
    ),
    n=1
)

toolbox.register('population', tools.initRepeat, list, toolbox.individual)
toolbox.register('evaluate', evaluate_individual)

def safe_crossover(ind1, ind2):
    """Realiza o crossover e corrige o parâmetro 'tile_size' se necessário."""
    tools.cxTwoPoint(ind1, ind2)
    for ind in [ind1, ind2]:
        if not isinstance(ind[5], tuple) or len(ind[5]) != 2:
            logging.warning(f"Correção pós-crossover: tile_size corrompido em {ind[5]}. Restaurando...")
            ind[5] = (random.choice([4, 8, 16]), random.choice([4, 8, 16]))
    return ind1, ind2

toolbox.register('mate', safe_crossover)

def mutate_individual(individual, indpb=0.2):
    """Realiza mutações aleatórias nos parâmetros do indivíduo."""
    if random.random() < indpb:
        individual[0] = random.randint(0, len(preproc_options) - 1)
    if random.random() < indpb:
        individual[1] = random.randint(0, 1)
    if random.random() < indpb:
        individual[2] = random.randint(90, 110)
    if random.random() < indpb:
        individual[3] = random.randint(130, 160)
    if random.random() < indpb:
        individual[4] = random.uniform(1.0, 4.0)
    if random.random() < indpb:
        individual[5] = (random.choice([4, 8, 16]), random.choice([4, 8, 16]))
    if random.random() < indpb:
        individual[6] = random.uniform(0.8, 1.5)
    if random.random() < indpb:
        individual[7] = random.uniform(2.0, 5.0)
    return (individual,)

toolbox.register('mutate', mutate_individual, indpb=0.2)
toolbox.register('select', tools.selTournament, tournsize=3)


In [ ]:
"""
Este script utiliza um Algoritmo Genético (GA) para otimizar os parâmetros do pipeline de pré‑processamento de imagens
e treinamento de um modelo YOLO. O pipeline inclui opções de equalização de histograma, CLAHE, correção de gamma e
remoção de fundo.
"""

# Inicializa a população com o tamanho definido em GA_POPULATION
population = toolbox.population(n=GA_POPULATION)

# Configuração das estatísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values[0])
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

def parallel_evaluation(pop):
    num_workers = min(2, os.cpu_count())
    fitnesses = Parallel(n_jobs=num_workers)(
        delayed(evaluate_individual)(ind) for ind in pop
    )
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit

# Avalia a população inicial
parallel_evaluation(population)

# Execução do GA por GA_GENERATIONS gerações
population, logbook = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=0.5,
    mutpb=0.3,
    ngen=GA_GENERATIONS,  # <-- Agora utilizando GA_GENERATIONS
    stats=stats,
    verbose=True
)

df_stats = pd.DataFrame(logbook)
print("Estatísticas por geração:")
print(df_stats)

df_evals = pd.DataFrame(ga_evaluations)
df_evals.to_csv("ga_detailed_evaluations.csv", index=False)

# Seleciona o melhor indivíduo
best_ind = tools.selBest(population, k=1)[0]
logging.info(f"Melhor configuração: {best_ind.details}")

# Salva os detalhes do melhor indivíduo em um arquivo JSON
with open("best_preproc_params.json", "w") as f:
    json.dump(best_ind.details, f)

print("Parâmetros do melhor pipeline de pré‑processamento salvos em 'best_preproc_params.json'.")


## Pré-processamento do dataset e treinamento do modelo YOLO
Este código realiza o pré-processamento das imagens do dataset de treino e validação antes de iniciar o treinamento
do modelo YOLO. Ele aplica técnicas de aprimoramento de imagem usando parâmetros otimizados e treina o modelo
com os dados pré-processados.

In [ ]:
# Diretórios de origem do dataset original
input_train_images_dir = "/content/dataset_original/train/images"
input_train_labels_dir = "/content/dataset_original/train/labels"
input_valid_images_dir = "/content/dataset_original/valid/images"
input_valid_labels_dir = "/content/dataset_original/valid/labels"

# Diretórios de destino para o novo dataset pré‑processado
output_train_images_dir = "/content/dataset_preprocessed/train/images"
output_train_labels_dir = "/content/dataset_preprocessed/train/labels"
output_valid_images_dir = "/content/dataset_preprocessed/valid/images"
output_valid_labels_dir = "/content/dataset_preprocessed/valid/labels"

# Cria os diretórios de destino, se ainda não existirem
for d in [output_train_images_dir, output_train_labels_dir, output_valid_images_dir, output_valid_labels_dir]:
    os.makedirs(d, exist_ok=True)

# Vamos carregar os melhores parâmetros encontrados pelo GA
best_params = {}
if os.path.exists("best_preproc_params.json"):
    with open("best_preproc_params.json", "r") as f:
        best_params = json.load(f)
else:
    # Caso não exista, definimos um dicionário básico
    best_params = {
        "preproc": "none",
        "remove_bg": False,
        "lower_blue": 100,
        "upper_blue": 140,
        "clahe_clip": 2.0,
        "tile_size": (8,8),
        "gamma_value": 1.0,
        "roi_scale": 3.0
    }

# Função de pré‑processamento utilizando os parâmetros otimizados
def preprocess_training_image(img):
    processed_img = apply_pipeline(
        image=img,
        use_hist_eq=(best_params["preproc"] == "hist_eq"),
        use_clahe=(best_params["preproc"] == "clahe"),
        clahe_clip=best_params["clahe_clip"],
        tile_size=tuple(best_params["tile_size"]) if isinstance(best_params["tile_size"], list) else best_params["tile_size"],
        use_gamma=(best_params["preproc"] == "gamma"),
        gamma_value=best_params["gamma_value"],
        use_remove_bg=best_params["remove_bg"],
        lower_bg=(best_params["lower_blue"], 100, 100),
        upper_bg=(best_params["upper_blue"], 255, 255)
    )
    return processed_img

# Processa e salva as imagens de treino
train_image_files = glob.glob(os.path.join(input_train_images_dir, "*.jpg"))
for img_file in train_image_files:
    img = cv2.imread(img_file)
    if img is None:
        continue
    processed_img = preprocess_training_image(img)
    base_name = os.path.basename(img_file)
    output_img_path = os.path.join(output_train_images_dir, base_name)
    cv2.imwrite(output_img_path, processed_img)

# Processa e salva as imagens de validação
valid_image_files = glob.glob(os.path.join(input_valid_images_dir, "*.jpg"))
for img_file in valid_image_files:
    img = cv2.imread(img_file)
    if img is None:
        continue
    processed_img = preprocess_training_image(img)
    base_name = os.path.basename(img_file)
    output_img_path = os.path.join(output_valid_images_dir, base_name)
    cv2.imwrite(output_img_path, processed_img)

print("Imagens pré‑processadas salvas com sucesso!")

# Copia os arquivos de labels do dataset original para o novo dataset pré‑processado
train_label_files = glob.glob(os.path.join(input_train_labels_dir, "*.txt"))
for label_file in train_label_files:
    base_name = os.path.basename(label_file)
    output_label_path = os.path.join(output_train_labels_dir, base_name)
    shutil.copy(label_file, output_label_path)

valid_label_files = glob.glob(os.path.join(input_valid_labels_dir, "*.txt"))
for label_file in valid_label_files:
    base_name = os.path.basename(label_file)
    output_label_path = os.path.join(output_valid_labels_dir, base_name)
    shutil.copy(label_file, output_label_path)

print("Labels copiadas com sucesso!")


Esta etapa final realiza o treinamento do modelo YOLO usando os dados previamente pré-processados.
O treinamento é configurado para utilizar a GPU, se disponível, e os hiperparâmetros podem ser ajustados
conforme necessário para otimizar o desempenho do modelo.

In [ ]:
# Em seguida, inicie o treinamento final utilizando o dataset pré‑processado.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = YOLO("yolov11-small.pt").to(device)

# Treinamento final (ajuste os parâmetros conforme necessário)
model.train(
    data="/content/dataset_original/original.yaml",  # ou o YAML atualizado com o caminho para o dataset pré‑processado
    epochs=TRAINING_EPOCHS,  # <-- Agora utilizando TRAINING_EPOCHS
    imgsz=1280,
    batch=16,
    device=device,
    augment=True
)

# Após o treinamento, salve o modelo final
final_model_path = "/content/final_yolov11.pt"

# Método 1: Utilizando o método save() do objeto YOLO (se disponível)
try:
    model.save(final_model_path)
    print("Modelo final salvo em:", final_model_path)
except Exception as e:
    print("Método model.save() não disponível ou falhou. Salvando via state_dict.")
    torch.save(model.model.state_dict(), final_model_path)
    print("Modelo final salvo via state_dict em:", final_model_path)

## Carregamento do modelo YOLO e processamento de vídeo

 Este trecho de código carrega um modelo YOLO para detecção de objetos e aplica-o a cada frame de um vídeo de entrada.
 O vídeo processado, com as detecções desenhadas, é salvo como um novo arquivo.


In [ ]:
# Carrega o modelo YOLO
model = YOLO("final_yolov11.pt")

# Abre o vídeo de entrada
cap = cv2.VideoCapture("video.mp4")
if not cap.isOpened():
    print("Erro ao abrir o video.mp4")
    exit()

# Recupera propriedades do vídeo
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output_no_preproc.mp4", fourcc, fps, (width, height))

# Variáveis para estatísticas
frame_count = 0
total_detections = 0

# Inicia a contagem de tempo para medir FPS de processamento
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    # Executa a inferência do modelo no frame com tamanho 1280x1280
    results = model.predict(frame, imgsz=1280)

    # Contar apenas as detecções de bola
    num_ball_detections = 0
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        cls_name = model.names[cls_id]
        if cls_name == "bola":
            num_ball_detections += 1

    total_detections += num_ball_detections

    # Obtém o frame anotado (o método .plot() retorna a imagem com as detecções desenhadas)
    annotated_frame = results[0].plot()

    # Escreve o frame processado no vídeo de saída
    out.write(annotated_frame)

# Finaliza tempo de processamento
end_time = time.time()
total_time = end_time - start_time

cap.release()
out.release()
cv2.destroyAllWindows()

# Estatísticas finais
processing_fps = frame_count / total_time if total_time > 0 else 0

print("========================================")
print("Processamento finalizado: output_no_preproc.mp4 gerado com sucesso!")
print(f"FPS do vídeo de entrada: {fps:.2f}")
print(f"Frames processados: {frame_count}")
print(f"Detecções de bola (total): {total_detections}")
print(f"FPS de processamento (estimado): {processing_fps:.2f}")
print("========================================")


Este código realiza a detecção e rastreamento da bola de futebol em um vídeo, aplicando pré-processamento otimizado
para melhorar a precisão do modelo YOLO treinado.

In [ ]:

##################################################
# 1) Lê os parâmetros otimizados
##################################################
try:
    with open("best_preproc_params.json", "r") as f:
        best_params = json.load(f)
except FileNotFoundError:
    logging.warning("Arquivo best_preproc_params.json não encontrado. Usando valores padrão.")
    best_params = {
        "preproc": "none",
        "remove_bg": False,
        "lower_blue": 100,
        "upper_blue": 140,
        "clahe_clip": 2.0,
        "tile_size": (8, 8),
        "gamma_value": 1.0
    }

##################################################
# 2) Configurações iniciais
##################################################
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

# Carrega o modelo YOLO (Ultralytics)
model = YOLO("yolov11-small.pt").to(device)

# Lê parâmetros do JSON
use_hist_eq   = (best_params["preproc"] == "hist_eq")
use_clahe     = (best_params["preproc"] == "clahe")
use_gamma     = (best_params["preproc"] == "gamma")
use_remove_bg = best_params["remove_bg"]

clahe_clip  = best_params.get("clahe_clip", 2.0)
tile_size   = tuple(best_params.get("tile_size", (8,8)))
gamma_value = best_params.get("gamma_value", 1.0)

lower_bg = (best_params.get("lower_blue", 100), 100, 100)  # HSV
upper_bg = (best_params.get("upper_blue", 140), 255, 255)  # HSV

##################################################
# 3) Abre o vídeo de entrada e prepara a saída
##################################################
input_video_path = "video.mp4"
output_video_path = "output_preproc.mp4"

cap = cv2.VideoCapture(input_video_path)
if not cap.isOpened():
    print(f"Erro ao abrir {input_video_path}")
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

##################################################
# 4) Processa frame a frame (sem ROI)
##################################################

frame_count = 0
total_detections = 0

start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break  # Fim do vídeo

    frame_count += 1
    annotated_frame = frame.copy()  # onde desenharemos as boxes

    # 4.1) Aplica pipeline de pré-processamento no frame inteiro
    preproc_frame = apply_pipeline(
        frame,
        use_hist_eq=use_hist_eq,
        use_clahe=use_clahe,
        clahe_clip=clahe_clip,
        tile_size=tile_size,
        use_gamma=use_gamma,
        gamma_value=gamma_value,
        use_remove_bg=use_remove_bg,
        lower_bg=lower_bg,
        upper_bg=upper_bg
    )

    # 4.2) Converte a imagem pré-processada em tensor e faz padding para múltiplos de 32
    frame_tensor = torch.tensor(preproc_frame, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device) / 255.0

    _, _, h, w = frame_tensor.shape
    new_h = math.ceil(h / 32) * 32
    new_w = math.ceil(w / 32) * 32
    pad_h = new_h - h
    pad_w = new_w - w
    frame_tensor = F.pad(frame_tensor, (0, pad_w, 0, pad_h), mode='constant', value=0)

    # 4.3) Inferência YOLO
    results = model(frame_tensor)

    # 4.4) Extrai a bounding box da bola (se existir)
    ball_box = extract_ball_box(results[0], preproc_frame)

    if ball_box is not None:
        total_detections += 1
        cx, cy, bw, bh = ball_box
        x1 = int(cx - bw / 2)
        y1 = int(cy - bh / 2)
        x2 = int(cx + bw / 2)
        y2 = int(cy + bh / 2)
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # 4.5) Escreve o frame anotado no vídeo de saída
    out.write(annotated_frame)

##################################################
# 5) Libera recursos e exibe estatísticas
##################################################
cap.release()
out.release()
cv2.destroyAllWindows()

end_time = time.time()
total_time = end_time - start_time
processing_fps = frame_count / total_time if total_time > 0 else 0

print("========================================")
print(f"Processamento finalizado. Vídeo salvo em '{output_video_path}'.")
print(f"FPS do vídeo de entrada: {fps:.2f}")
print(f"Frames processados: {frame_count}")
print(f"Frames com bola detectada: {total_detections}")
print(f"FPS de processamento (estimado): {processing_fps:.2f}")
print("========================================")

Este código realiza a detecção e rastreamento da bola de futebol em um vídeo, aplicando pré-processamento otimizado
para melhorar a precisão do modelo YOLO treinado. A detecção ocorre inicialmente no vídeo completo e, uma vez encontrada,
uma região de interesse (ROI) é utilizada para otimizar a detecção nos quadros seguintes.



In [ ]:
##################################################
# 1) Lê os parâmetros otimizados
##################################################
try:
    with open("best_preproc_params.json", "r") as f:
        best_params = json.load(f)
except FileNotFoundError:
    logging.warning("Arquivo best_preproc_params.json não encontrado. Usando valores padrão.")
    best_params = {
        "preproc": "none",
        "remove_bg": False,
        "lower_blue": 100,
        "upper_blue": 140,
        "clahe_clip": 2.0,
        "tile_size": (8, 8),
        "gamma_value": 1.0,
        "roi_scale": 3.0
    }

##################################################
# 2) Configurações iniciais
##################################################
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

# Carrega o modelo YOLO (Ultralytics)
model = YOLO("yolov11-small.pt").to(device)

# Lê parâmetros do JSON
use_hist_eq   = (best_params["preproc"] == "hist_eq")
use_clahe     = (best_params["preproc"] == "clahe")
use_gamma     = (best_params["preproc"] == "gamma")
use_remove_bg = best_params["remove_bg"]

clahe_clip  = best_params.get("clahe_clip", 2.0)
tile_size   = tuple(best_params.get("tile_size", (8,8)))
gamma_value = best_params.get("gamma_value", 1.0)
roi_scale   = best_params.get("roi_scale", 3.0)

lower_bg = (best_params.get("lower_blue", 100), 100, 100)  # HSV
upper_bg = (best_params.get("upper_blue", 140), 255, 255)  # HSV

##################################################
# 3) Abre o vídeo de entrada e prepara a saída
##################################################
input_video_path = "video.mp4"
output_video_path = "output_preproc_roi.mp4"

cap = cv2.VideoCapture(input_video_path)
if not cap.isOpened():
    print(f"Erro ao abrir {input_video_path}")
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# Última bounding box conhecida (para ROI)
last_ball_bb = None

##################################################
# 4) Processa frame a frame
##################################################

# Variáveis para estatísticas
frame_count = 0
total_detections = 0

# Tempo inicial
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break  # Fim do vídeo

    frame_count += 1
    annotated_frame = frame.copy()  # onde desenharemos as boxes

    # 4.1) Se existir última posição conhecida, recorta ROI
    if last_ball_bb is not None:
        roi_coords = get_roi_from_bbox(last_ball_bb, frame.shape, roi_scale)
        roi_frame = frame[
            roi_coords["top"]:roi_coords["bottom"],
            roi_coords["left"]:roi_coords["right"]
        ]
    else:
        # Sem referência anterior, usa frame inteiro
        roi_frame = frame

    # 4.2) Aplica pipeline de pré-processamento
    preproc_roi = apply_pipeline(
        roi_frame,
        use_hist_eq=use_hist_eq,
        use_clahe=use_clahe,
        clahe_clip=clahe_clip,
        tile_size=tile_size,
        use_gamma=use_gamma,
        gamma_value=gamma_value,
        use_remove_bg=use_remove_bg,
        lower_bg=lower_bg,
        upper_bg=upper_bg
    )

    # 4.3) ROI -> tensor, faz padding para múltiplos de 32 e inferência
    roi_tensor = torch.tensor(preproc_roi, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device) / 255.0

    _, _, h, w = roi_tensor.shape
    new_h = math.ceil(h / 32) * 32
    new_w = math.ceil(w / 32) * 32
    pad_h = new_h - h
    pad_w = new_w - w

    roi_tensor = F.pad(roi_tensor, (0, pad_w, 0, pad_h), mode='constant', value=0)

    # Inferência YOLO
    results = model(roi_tensor)

    # Extrai a bounding box da bola dentro da ROI
    ball_box = extract_ball_box(results[0], preproc_roi)

    # 4.4) Ajusta coords pro frame completo ou força fallback
    if ball_box is not None:
        cx, cy, bw, bh = ball_box
        if last_ball_bb is not None:
            # Ajustar coordenadas caso seja ROI
            cx += roi_coords["left"]
            cy += roi_coords["top"]
        last_ball_bb = (cx, cy, bw, bh)
    else:
        # Se não encontrou no ROI, marca last_ball_bb como None
        last_ball_bb = None

    # 4.5) Se não encontramos no ROI (last_ball_bb = None), fallback para a imagem inteira
    if last_ball_bb is None:
        full_preproc = apply_pipeline(
            frame,
            use_hist_eq=use_hist_eq,
            use_clahe=use_clahe,
            clahe_clip=clahe_clip,
            tile_size=tile_size,
            use_gamma=use_gamma,
            gamma_value=gamma_value,
            use_remove_bg=use_remove_bg,
            lower_bg=lower_bg,
            upper_bg=upper_bg
        )
        full_tensor = torch.tensor(full_preproc, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device) / 255.0

        _, _, fh, fw = full_tensor.shape
        new_fh = math.ceil(fh / 32) * 32
        new_fw = math.ceil(fw / 32) * 32
        pad_fh = new_fh - fh
        pad_fw = new_fw - fw
        full_tensor = F.pad(full_tensor, (0, pad_fw, 0, pad_fh), mode='constant', value=0)

        results_full = model(full_tensor)
        ball_box_fallback = extract_ball_box(results_full[0], full_preproc)
        if ball_box_fallback is not None:
            last_ball_bb = ball_box_fallback

    # 4.6) Se detectamos alguma bola, incrementamos contagem e desenha bounding box
    if last_ball_bb is not None:
        total_detections += 1

        cx, cy, bw, bh = last_ball_bb
        x1 = int(cx - bw / 2)
        y1 = int(cy - bh / 2)
        x2 = int(cx + bw / 2)
        y2 = int(cy + bh / 2)
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # 4.7) Escreve o frame anotado no vídeo de saída
    out.write(annotated_frame)

##################################################
# 5) Libera recursos e exibe estatísticas
##################################################
cap.release()
out.release()
cv2.destroyAllWindows()

end_time = time.time()
total_time = end_time - start_time
processing_fps = frame_count / total_time if total_time > 0 else 0

print("========================================")
print(f"Processamento finalizado. Vídeo salvo em '{output_video_path}'.")
print(f"FPS do vídeo de entrada: {fps:.2f}")
print(f"Frames processados: {frame_count}")
print(f"Frames com bola detectada: {total_detections}")
print(f"FPS de processamento (estimado): {processing_fps:.2f}")
print("========================================")

## 5. Conclusões Finais

Neste notebook foi implementado um pipeline completo para a detecção da bola em partidas, englobando as seguintes etapas:

- Preparação dos dados e extração dos frames do vídeo;
- Aplicação de diversas técnicas de pré‑processamento (equalização, CLAHE, correção gamma e remoção de fundo);
- Criação do arquivo YAML para definição do dataset;
- Treinamento do modelo YOLO com diferentes configurações e otimização dos parâmetros via Algoritmo Genético.

Os resultados de cada configuração foram registrados e salvos em arquivos de log e CSV, permitindo a análise comparativa entre as abordagens.

**FIM DO NOTEBOOK**